# Step 1: Library Imports and Device Configuration

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive

# Mount Google Drive to access the dataset paths
drive.mount('/content/drive', force_remount=True)

# Set working directory
project_dir = "/content/drive/MyDrive/Brain_Tumor_Classification"
%cd {project_dir}

# Device configuration (Use GPU/CUDA if available for fast training)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1Rd5_epoDI2porCXmS9-4opjNV4Tdx9s2/Brain_Tumor_Classification
Using Device: cpu


# Step 2: CNN Architecture Design (Baseline Model)

In [2]:
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(BrainTumorCNN, self).__init__()

        # Block 1: Input (3, 224, 224) -> Output (16, 112, 112)
        self.features1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2: Input (16, 112, 112) -> Output (32, 56, 56)
        self.features2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Block 3: Input (32, 56, 56) -> Output (64, 28, 28)
        self.features3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Fully Connected Layers: Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5), # Prevents overfitting
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features1(x)
        x = self.features2(x)
        x = self.features3(x)
        x = self.classifier(x)
        return x

# Instantiate the model and move it to GPU
model = BrainTumorCNN(num_classes=4).to(device)
print(model)

BrainTumorCNN(
  (features1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (features2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (features3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=501

# Step 3: Loss Function, Optimizer, and Training Pipeline Definition

In [3]:
# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Function Template for 1 Epoch to sanity check the baseline
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    print("🏋️ Training baseline model for a sanity check epoch...")
    # This loop simulates processing the DataLoader batches
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    print(f"📊 Sanity Check Result -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")
    return epoch_loss, epoch_acc

print("Step 3 Baseline Model Pipeline compiled without errors!")

Step 3 Baseline Model Pipeline compiled without errors!
